# 99 · Orchestrate one full run

Runs stages 01-06 as a DAG with `notebookutils.notebook.runMultiple`, passing one `run_id` to every stage.

Attach the **pipeline Lakehouse** as default and the **goldenrecord Environment** (wheel + config). All logic is in the `goldenrecord` library; this notebook only calls it.

In [ ]:
# Parameters (overridden by the pipeline / orchestrator)
run_id = ""                                      # same id for every stage of one run
config_dir = "/lakehouse/default/Files/config"   # config/ folder uploaded to the Lakehouse
landing_root = "Files/landing"                   # where source extracts land
use_key_vault = True                             # False only in a sandbox without Key Vault
use_fabric_ai = True                             # False -> deterministic heuristic (no AI calls)

In [ ]:
import uuid
from datetime import datetime, timezone

run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
args = {
    "run_id": run_id,
    "config_dir": config_dir,
    "landing_root": landing_root,
    "use_key_vault": use_key_vault,
    "use_fabric_ai": use_fabric_ai,
}


def activity(name, depends_on=()):
    return {
        "name": name,
        "path": name,
        "timeoutPerCellInSeconds": 3600,
        "args": args,
        "dependencies": list(depends_on),
    }


dag = {
    "activities": [
        activity("01_ingest_bronze"),
        activity("02_cleanse_silver", ["01_ingest_bronze"]),
        activity("03_match_candidates", ["02_cleanse_silver"]),
        activity("04_publish_review_queue", ["03_match_candidates"]),
        activity("05_resolve_publish_gold", ["04_publish_review_queue"]),
        activity("06_evaluate_matching", ["05_resolve_publish_gold"]),
    ],
    "timeoutInSeconds": 4 * 3600,
    "concurrency": 2,
}
results = notebookutils.notebook.runMultiple(dag, {"displayDAGViaGraphviz": False})
results